# 07 — Production Context Manager — Practical Examples

**Covers**: Full ProductionContextManager class, agent loop integration, session persistence, metrics dashboard, multi-strategy demonstration

In [ ]:
import os, json, time
import tiktoken
import numpy as np
from openai import OpenAI
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. ProductionContextManager — Full Class

In [ ]:
@dataclass
class ContextMetrics:
    total_added:    int = 0
    api_calls:      int = 0
    compressions:   int = 0
    messages_dropped: int = 0
    token_history:  list = field(default_factory=list)

    def update(self, tokens: int):
        self.api_calls += 1
        self.token_history.append(tokens)

    def summary(self) -> dict:
        h = self.token_history
        return {'api_calls': self.api_calls, 'compressions': self.compressions,
                'messages_dropped': self.messages_dropped,
                'avg_tokens': round(sum(h)/len(h)) if h else 0,
                'peak_tokens': max(h, default=0)}


class ProductionContextManager:
    """
    Production-grade context manager:
    - Pins system prompt
    - Auto-compresses via LLM summarization when utilization > target
    - Serializable for session persistence
    - Full metrics tracking
    """

    def __init__(self, system: str, context_limit: int = 128_000,
                 output_reserve: int = 4_096, target_util: float = 0.55,
                 model: str = MODEL, session_id: str = None):
        self.system          = system
        self.context_limit   = context_limit
        self.output_reserve  = output_reserve
        self.target_util     = target_util
        self.model           = model
        self.session_id      = session_id or datetime.now().strftime('%Y%m%d_%H%M%S')
        self.enc             = tiktoken.encoding_for_model(model)
        self.metrics         = ContextMetrics()
        self._history:  list[dict] = []
        self._summary:  Optional[str] = None

    def add(self, role: str, content: str):
        self._history.append({'role': role, 'content': content})
        self.metrics.total_added += 1
        if self._utilization() > self.target_util:
            self._compress()

    def get_messages(self) -> list[dict]:
        msgs = [{'role': 'system', 'content': self.system}]
        if self._summary:
            msgs.append({'role': 'assistant', 'content': f'[Summary of earlier conversation]\n{self._summary}'})
        msgs += self._trim_history()
        self.metrics.update(self._count_msgs(msgs))
        return msgs

    def stats(self) -> dict:
        m = self.metrics.summary()
        m['history_msgs']   = len(self._history)
        m['utilization_pct']= round(self._utilization() * 100, 1)
        m['has_summary']    = self._summary is not None
        m['session_id']     = self.session_id
        return m

    def to_json(self) -> str:
        return json.dumps({'session_id': self.session_id, 'system': self.system,
                           'history': self._history, 'summary': self._summary,
                           'config': {'context_limit': self.context_limit,
                                      'output_reserve': self.output_reserve,
                                      'target_util': self.target_util, 'model': self.model}}, indent=2)

    @classmethod
    def from_json(cls, s: str) -> 'ProductionContextManager':
        d = json.loads(s)
        c = d['config']
        m = cls(d['system'], c['context_limit'], c['output_reserve'], c['target_util'], c['model'], d['session_id'])
        m._history, m._summary = d['history'], d['summary']
        return m

    # ── Private ─────────────────────────────────────────────────────────
    def _count(self, text: str) -> int:
        return len(self.enc.encode(str(text)))
    def _count_msgs(self, msgs: list[dict]) -> int:
        return sum(3 + self._count(m.get('content','')) for m in msgs) + 3
    def _utilization(self) -> float:
        sys_t = 3 + self._count(self.system)
        sum_t = (3 + self._count(self._summary)) if self._summary else 0
        his_t = self._count_msgs(self._history)
        return (sys_t + sum_t + his_t + self.output_reserve) / self.context_limit
    def _trim_history(self) -> list[dict]:
        sys_t = 3 + self._count(self.system) + 3
        sum_t = (3 + self._count(self._summary)) if self._summary else 0
        budget = int(self.context_limit * self.target_util) - sys_t - sum_t - self.output_reserve
        kept, used = [], 0
        for m in reversed(self._history):
            t = 3 + self._count(m.get('content',''))
            if used + t <= budget: kept.append(m); used += t
            else: self.metrics.messages_dropped += 1
        return list(reversed(kept))
    def _compress(self):
        if len(self._history) < 4: return
        split = len(self._history) // 2
        old = self._history[:split]
        self._history = self._history[split:]
        text = '\n'.join(f"{m['role'].upper()}: {m['content']}" for m in old if m['role'] != 'system')
        prompt = (f'Update this summary:\n{self._summary}\n\nNew:\n{text}' if self._summary
                  else f'Summarize this conversation (2 paragraphs):\n{text}')
        r = client.chat.completions.create(model=self.model,
                                           messages=[{'role': 'user', 'content': prompt}],
                                           max_tokens=300, temperature=0.0)
        self._summary = r.choices[0].message.content
        self.metrics.compressions += 1
        print(f'  🗜️  Compression #{self.metrics.compressions} ({len(old)} messages compressed)')

print('ProductionContextManager ready ✅')

## 2. Agent Loop Demo — 10 Turns with Automatic Context Management

In [ ]:
ctx = ProductionContextManager(
    system='You are a knowledgeable Python tutor. Explain concepts clearly with short examples.',
    target_util=0.40,  # Low threshold to trigger compression early for demo
    model=MODEL
)

topics = [
    'What is a Python decorator?',
    'Show me a simple decorator example.',
    'What is the difference between @staticmethod and @classmethod?',
    'Explain Python context managers with the `with` statement.',
    'What are Python generators and when should I use them?',
    'How does yield differ from return?',
    'Explain async/await in Python.',
    'What is the asyncio event loop?',
    'How do I run async code from synchronous code?',
    'Give me a complete async example with aiohttp.',
]

print(f"Running {len(topics)}-turn Python tutoring session")
print('─' * 65)

for i, question in enumerate(topics, 1):
    ctx.add('user', question)
    msgs = ctx.get_messages()
    r = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=100)
    answer = r.choices[0].message.content
    ctx.add('assistant', answer)
    s = ctx.stats()
    print(f"Turn {i:>2}: {question[:45]:<47} | {s['utilization_pct']:>5}% | summary={'✅' if s['has_summary'] else '❌'}")

print()
print('Final stats:', ctx.stats())

## 3. Metrics Dashboard

In [ ]:
def print_dashboard(ctx: ProductionContextManager):
    s = ctx.stats()
    pct = s['utilization_pct']
    bar_n = int(pct / 100 * 40)
    bar = '█' * bar_n + '░' * (40 - bar_n)
    icon = '🟢' if pct < 55 else ('🟡' if pct < 75 else '🔴')
    print(f"\n{'═'*55}")
    print(f"  Context Manager Dashboard ({ctx.session_id})")
    print(f"{'═'*55}")
    print(f"  [{bar}] {pct}% {icon}")
    print(f"  {'Session messages:':<28} {s['total_added']}")
    print(f"  {'API calls made:':<28} {s['api_calls']}")
    print(f"  {'Compressions triggered:':<28} {s['compressions']}")
    print(f"  {'Messages dropped:':<28} {s['messages_dropped']}")
    print(f"  {'Avg tokens/call:':<28} {s['avg_tokens']:,}")
    print(f"  {'Peak tokens/call:':<28} {s['peak_tokens']:,}")
    print(f"  {'Has summary:':<28} {s['has_summary']}")
    print(f"{'═'*55}")

print_dashboard(ctx)

## 4. Session Persistence — Save and Restore

In [ ]:
import tempfile, os

# Save session
state_json = ctx.to_json()
save_path = os.path.join(tempfile.gettempdir(), f'session_{ctx.session_id}.json')
with open(save_path, 'w') as f:
    f.write(state_json)
print(f'💾 Saved session to {save_path} ({len(state_json):,} chars)')

# Restore session
with open(save_path) as f:
    restored = ProductionContextManager.from_json(f.read())

print(f'\n✅ Restored session: {restored.session_id}')
print(f'   History: {len(restored._history)} messages')
print(f'   Has summary: {restored._summary is not None}')

# Continue the restored session
restored.add('user', 'Based on everything we covered, what should I learn next?')
msgs = restored.get_messages()
r = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=150)
print(f'\nResumed answer: {r.choices[0].message.content[:200]}')

## 5. Comparing All Context Management Strategies Side-By-Side

In [ ]:
enc = tiktoken.encoding_for_model(MODEL)

# Simulate 10 synthetic turns
def make_history(n: int) -> list[dict]:
    h = []
    for i in range(n):
        h.append({'role': 'user',      'content': f'Question {i+1}: Explain Python concept #{i+1} in detail.'})
        h.append({'role': 'assistant', 'content': f'Answer {i+1}: Python concept #{i+1} involves several key ideas including X, Y, and Z. Here is a code example...' * 3})
    return h

hist = make_history(10)
raw_tokens = sum(3 + len(enc.encode(m.get('content',''))) for m in hist) + 3

# Strategy 1: Sliding window (keep last 8 messages)
sliding = [{'role': 'system', 'content': 'You are helpful.'}] + hist[-8:]
sliding_t = sum(3 + len(enc.encode(m.get('content',''))) for m in sliding) + 3

# Strategy 2: Summarization (simulate: compress first 12 messages to ~100 tokens)
summary_placeholder = 'This conversation covered Python concepts 1-6 including variables, lists, dicts, loops, functions, and classes with examples.'
with_summary = [{'role': 'system', 'content': 'You are helpful.'},
                {'role': 'assistant', 'content': f'[Summary]\n{summary_placeholder}'}] + hist[-4:]
summary_t = sum(3 + len(enc.encode(m.get('content',''))) for m in with_summary) + 3

# Strategy 3: RAG (simulate: 2 retrieved messages + 4 recent)
ret_ctx = [{'role': 'user',      'content': 'Relevant past: Q3 about lists, Q7 about functions.'},
           {'role': 'assistant', 'content': 'I recall these topics.'}]
rag_msgs = [{'role': 'system', 'content': 'You are helpful.'}] + ret_ctx + hist[-4:]
rag_t = sum(3 + len(enc.encode(m.get('content',''))) for m in rag_msgs) + 3

print(f"{'Strategy':<25} {'Tokens':>8} {'vs Raw':>8} {'Msgs':>6} {'What's kept'}")
print('-' * 80)
print(f"{'Raw (all history)':<25} {raw_tokens:>8,} {'100%':>8} {len(hist):>6} All 10 turns")
print(f"{'Sliding (last 8 msgs)':<25} {sliding_t:>8,} {sliding_t/raw_tokens:>8.0%} {len(sliding):>6} Last 4 turns only")
print(f"{'Summary + 2 turns':<25} {summary_t:>8,} {summary_t/raw_tokens:>8.0%} {len(with_summary):>6} Summary + last 2 turns")
print(f"{'RAG + 2 turns':<25} {rag_t:>8,} {rag_t/raw_tokens:>8.0%} {len(rag_msgs):>6} Relevant + last 2 turns")

## 📌 Final Summary — All 7 Topics in One View

| Topic | Key Pattern | Code |
|---|---|---|
| 01 Token limits | Count before send | `tiktoken.encoding_for_model(model)` |
| 02 Sliding window | Keep last N tokens | `ContextWindow(system, max_tokens)` |
| 03 Summarization | Compress old history | `RollingSummaryContext` |
| 04 RAG context | Embed + retrieve | `SimpleVectorStore + cosine_sim` |
| 05 Token budgeting | Allocate per section | `TokenBudgetEnforcer` |
| 06 Long documents | Chunk + map-reduce | `chunk_with_overlap() + map_reduce()` |
| 07 Production | All strategies combined | `ProductionContextManager` |

**The production pattern**: `ctx.add()` → `ctx.get_messages()` → API call → repeat.